# FGPT AutoDiff & JaxConverter — Demo Notebook

This notebook demonstrates the JAX conversion layer using a **self-contained dummy NumPy class** that does not require the ORCHIDEE codebase or prior Fortran transpilation.

The dummy class mimics the structure of a real transpiled FGPT output:
- Class attributes representing physical fields (arrays and scalars)
- A method with a `for` loop over grid points
- An `if / else` conditional branch
- In-place array updates
- A call to a helper method

## What this notebook covers

1. [Setup](#1-setup)
2. [Write a dummy NumPy class](#2-write-a-dummy-numpy-class) — the kind F2NP + Transformer produce
3. [Inspect what AutoDiff does](#3-inspect-what-autodiff-does) — class-level restructuring
4. [Inspect what JaxConverter does](#4-inspect-what-jaxconverter-does) — control-flow rewriting
5. [Run the converted JAX module](#5-run-the-converted-jax-module) — verify numerical equivalence
6. [Explore the three modes](#6-explore-the-three-modes) — jax / fwd / bwd
7. [Cleanup](#7-cleanup)

---
## 1. Setup

In [1]:
%reload_ext autoreload
%autoreload 2

import os
import ast
import shutil
import tempfile
from pathlib import Path
from fgpt.core.common import Logger

logger = Logger()
# Ensure we run from project root template.yaml are found 
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

logger.info(f"Working directory : {os.getcwd()}")
logger.info(f"template.yaml     : {os.path.exists('template.yaml')}")

# Output directory for generated files
OUTPUT_DIR = tempfile.mkdtemp(prefix="fgpt_autodiff_demo_")
logger.info(f"Output directory  : {OUTPUT_DIR}")

[INFO] Working directory : /home/kardaneh/Fgpt

[INFO] template.yaml     : True

[INFO] Output directory  : /tmp/fgpt_autodiff_demo_whu5k4hc

---
## 2. Write a dummy NumPy class

This class mimics what `F2NP` + `Transformer` produce after transpiling a Fortran subroutine.

It contains the key constructs that `JaxConverter` needs to handle:

| Construct | JAX transformation |
|---|---|
| `for ji in range(...)` | `lax.scan` |
| `if cond: x = a else: x = b` | `jnp.where` |
| `if cond: <stateful block>` | `lax.cond` |
| `self.arr[ji] = val` | `self.arr = self.arr.at[ji].set(val)` |
| `np.zeros / np.ones` | `jnp.zeros / jnp.ones` |

In [2]:
# This is the kind of file that global_module_<routine>.py looks like
# after the Fortran-to-Python transpilation stage

GLOBAL_MODULE_SOURCE = '''\
import numpy as np


class DemoSoil:

    def __init__(self):
        # Scalar variables
        self.kjpindex = np.int32(10)
        self.dt = np.float64(1800.0)
        self.mc_sat = np.float64(0.4)
        self.mc_wilt = np.float64(0.1)

        # Arrays
        self.mc = np.zeros((self.kjpindex, ), dtype=np.float64)
        self.temp = np.zeros((self.kjpindex, ), dtype=np.float64)

    def compute_temperature(self):
        for ji in range(0, self.kjpindex, 1):
            self.temp[ji] = self.temp[ji] + 0.1 * self.dt * self.mc[ji]

    def compute_moisture(self, flux):

        for ji in range(0, self.kjpindex, 1):
            self.mc[ji] = self.mc[ji] + 0.01 * self.dt
            if self.mc[ji] < self.mc_sat:
                self.mc[ji] = self.mc[ji]
            else:
                self.mc[ji] = self.mc_sat

            if self.mc[ji] > self.mc_wilt:
                flux[ji] = (self.mc[ji] - self.mc_wilt) * self.dt
            
        self.compute_temperature()
'''

# Write the global module file
global_module_path = os.path.join(OUTPUT_DIR, "global_module_demo_soil.py")
with open(global_module_path, "w") as f:
    f.write(GLOBAL_MODULE_SOURCE)

logger.info(f"Written: {global_module_path}")

[INFO] Written: /tmp/fgpt_autodiff_demo_whu5k4hc/global_module_demo_soil.py

In [3]:
# This is the kind of file that main_<routine>.py looks like
# It instantiates the class, sets inputs, calls run(), and reads outputs

MAIN_SOURCE = '''\
import numpy as np
from global_module_demo_soil import DemoSoil


def main():
    model = DemoSoil()
    flux = np.zeros((model.kjpindex, ), dtype=np.float64)
    model.compute_moisture(flux)


if __name__ == "__main__":
    main()
'''

main_path = os.path.join(OUTPUT_DIR, "main_demo_soil.py")
with open(main_path, "w") as f:
    f.write(MAIN_SOURCE)

logger.info(f"Written: {main_path}")

[INFO] Written: /tmp/fgpt_autodiff_demo_whu5k4hc/main_demo_soil.py

In [4]:
import numpy as np
import sys

# Add output directory to the Python path
sys.path.insert(0, OUTPUT_DIR)

from global_module_demo_soil import DemoSoil

# Instantiate the translated module
model_np = DemoSoil()
flux = np.zeros((model_np.kjpindex, ), dtype=np.float64)
# Run translated NumPy implementation
model_np.compute_moisture(flux)

# Store reference outputs
ref_mc = model_np.mc.copy()
ref_flux = flux
ref_temp = model_np.temp.copy()

logger.info("NumPy reference outputs:")
logger.info(f"  mc   (soil moisture) : {ref_mc.round(4)}")
logger.info(f"  flux (drainage)      : {ref_flux.round(4)}")
logger.info(f"  temp (temperature)   : {ref_temp.round(2)}")

[INFO] NumPy reference outputs:

[INFO]   mc   (soil moisture) : [0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4]

[INFO]   flux (drainage)      : [540. 540. 540. 540. 540. 540. 540. 540. 540. 540.]

[INFO]   temp (temperature)   : [72. 72. 72. 72. 72. 72. 72. 72. 72. 72.]

## 4. Inspect what AutoDiff does

`AutoDiff` handles **class-level restructuring** — it's the JAX equivalent of `Transformer` in Stage 2:

- Rewrites `class DemoSoil:` → `class DemoSoil(eqx.Module):`
- Converts `np.*` → `jnp.*`
- Classifies attributes as static or dynamic Equinox fields
- Strips `logging` calls before tracing begins
- Delegates all control-flow rewriting to `JaxConverter`

In [5]:
from fgpt.autodiff import AutoDiff

# Instantiate AutoDiff — config_path=None uses the bundled default template
autodiff = AutoDiff(
    config_path=None,       # uses DEFAULT_TEMPLATE from src/fgpt/templates/default.yaml
    vectorize=['kjpindex'], # Defines the lower bound loop variable to vectorize
    mode="jax",             # produces _jax.py output
)

logger.info(f"Mode           : {autodiff.mode}")
logger.info(f"Config path    : {autodiff.config_path}")
logger.info(f"Benchmark dir  : {autodiff.benchmark_dir}")

/tmp/ipykernel_2249576/3093456184.py:4: UserWarning: No logger provided; using default Logger()
  autodiff = AutoDiff(


╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: AutoDiff                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Mode           : jax

[INFO] Config path    : /home/kardaneh/Fgpt/src/fgpt/templates/default.yaml

[INFO] Benchmark dir  : /home/kardaneh/Fgpt/benchmark

In [6]:
# Run the full conversion
autodiff.transform(
    class_file=global_module_path,
    main_file=main_path,
)

# Show what was produced
logger.info("\nGenerated files:")
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    size = f.stat().st_size
    logger.info(f"  {f.name:<50} ({size:>6} bytes)")

╭───────────────────────────────────────────────── 🚀 Node Start ─────────────────────────────────────────────────╮
│ Entering node: Create Module                                                                                    │
│ Function: transform(main_file='/tmp/fgpt_autodiff_demo_whu5k4hc/main_demo_soil.py',                             │
│ class_file='/tmp/fgpt_autodiff_demo_whu5k4hc/global_module_demo_soil.py', routine_dir=None)                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: JaxConverter                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Transforming procedure: compute_temperature

╭───────────────────────────────────────────────── 🚀 Node Start ─────────────────────────────────────────────────╮
│ Entering node: Transforming Python to JAX                                                                       │
│ 🔁 Converting function: compute_temperature                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── ✅ Node Complete ────────────────────────────────────────────────╮
│ Exiting node: Transforming Python to JAX                                                                        │
│ Duration: 0.00s                                                                                                 │
│ ✅ compute_temperature converted to JAX                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[INFO]   __pycache__                                        (    53 bytes)

[INFO]   global_module_demo_soil.py                         (   985 bytes)

[INFO]   global_module_demo_soil_jax.py                     (  1331 bytes)

[INFO]   main_demo_soil.py                                  (   232 bytes)

[INFO]   main_demo_soil_jax.py                              (  1065 bytes)

In [7]:
jax_global = os.path.join(OUTPUT_DIR, "main_demo_soil_jax.py")
logger.info("main_demo_soil_jax.py")
logger.info(f"\n{Path(jax_global).read_text()}")

[INFO] main_demo_soil_jax.py

[INFO] 
#!/usr/bin/env python3
import equinox as eqx
import jax.numpy as jnp
import jax
import numpy as np
from global_module_demo_soil_jax import DemoSoil_eqx
import time

def timer(func, *args):
    start = time.perf_counter()
    out = func(*args)
    jax.tree_util.tree_map(lambda x: x.block_until_ready() if hasattr(x, 'block_until_ready') else x, out)
    end = time.perf_counter()
    name = getattr(func, '__name__', func.func.__name__)
    duration = end - start
    print(f"[TIMER JAX] '{name}' executed in {duration:.6f}s")
    path = '/home/kardaneh/Fgpt/benchmark/{name}/time.txt'
    with open(path, 'a') as f:
        f.write(f"[TIMER JAX] Python: '{name}' executed in {duration:.6f} seconds\n")
        print(f'Saved the benchmark in:{path}')
    return out

def main():
    de = DemoSoil_eqx()
    jax.config.update('jax_enable_x64', True)
    flux = np.zeros((de.kjpindex,), dtype=np.float64)
    flux_jax = jnp.asarray(flux)
    de.compute_moisture(flux_jax)
    (flux_jax, de) = timer(de.compute_moisture, flux_jax)
if __name__ == '__main__':
    main()

In [8]:
# Read and display the converted JAX global module
jax_global = os.path.join(OUTPUT_DIR, "global_module_demo_soil_jax.py")

logger.info("global_module_demo_soil_jax.py")
logger.info(f"\n{Path(jax_global).read_text()}")

[INFO] global_module_demo_soil_jax.py

[INFO] 
#!/usr/bin/env python3
from jax import jit, lax
import equinox as eqx
import jax.numpy as jnp
import jax
import numpy as np

class DemoSoil_eqx(eqx.Module):
    kjpindex: int = eqx.field(static=True)
    dt: float
    mc_sat: float
    mc_wilt: float
    mc: jnp.ndarray
    temp: jnp.ndarray

    def __init__(self):
        self.kjpindex = 10
        self.dt = jnp.float64(1800.0)
        self.mc_sat = jnp.float64(0.4)
        self.mc_wilt = jnp.float64(0.1)
        self.mc = jnp.zeros((self.kjpindex,), dtype=jnp.float64)
        self.temp = jnp.zeros((self.kjpindex,), dtype=jnp.float64)

    def compute_temperature(self):
        temp = self.temp.at[:].set(self.temp + 0.1 * self.dt * self.mc)
        return (temp,)

    @eqx.filter_jit
    def compute_moisture(self, flux):
        mc = self.mc.at[:].set(self.mc + 0.01 * self.dt)
        _mask_0 = jnp.less(mc, self.mc_sat)
        mc = mc.at[:].set(jnp.where(_mask_0, mc, self.mc_sat))
        _mask_1 = jnp.greater(mc, self.mc_wilt)
        flux = flux.at[:].set(jnp.where(_mask_1, (mc - self.mc_wilt) * self.dt, flux[:]))
        self = eqx.tree_at(lambda m: (m.mc,), self, (mc,))
        (temp,) = self.compute_temperature()
        self = eqx.tree_at(lambda m: (m.temp,), self, (temp,))
        return (flux, eqx.tree_at(lambda m: (m.mc, m.temp), self, (mc, temp)))

---
## 5. Inspect what JaxConverter does

`JaxConverter` handles all **control-flow and expression rewriting** inside each method.

Let's look at the AST diff between the original and converted `compute_moisture` method to see exactly what changed.

In [9]:
# Parse both files and extract the compute_moisture method

def get_method_source(filepath, method_name):
    """Extract a method's source from a Python file using AST."""
    tree = ast.parse(Path(filepath).read_text())
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            for item in node.body:
                if isinstance(item, ast.FunctionDef) and item.name == method_name:
                    return ast.unparse(item)
    return None

# Original NumPy version
original = get_method_source(global_module_path, "compute_moisture")
logger.info("BEFORE — compute_moisture (NumPy)")
logger.info(f"\n{original}")

[INFO] BEFORE — compute_moisture (NumPy)

[INFO] 
def compute_moisture(self, flux):
    for ji in range(0, self.kjpindex, 1):
        self.mc = self.mc + 0.01 * self.dt
        if self.mc < self.mc_sat:
            self.mc = self.mc
        else:
            self.mc = self.mc_sat
        if self.mc > self.mc_wilt:
            flux = (self.mc - self.mc_wilt) * self.dt
    self.compute_temperature()

In [10]:
# Converted JAX version
converted = get_method_source(jax_global, "compute_moisture")
logger.info("AFTER — compute_moisture (JAX)")
logger.info(f"\n{converted}")

[INFO] AFTER — compute_moisture (JAX)

[INFO] 
@eqx.filter_jit
def compute_moisture(self, flux):
    mc = self.mc.at[:].set(self.mc + 0.01 * self.dt)
    _mask_0 = jnp.less(mc, self.mc_sat)
    mc = mc.at[:].set(jnp.where(_mask_0, mc, self.mc_sat))
    _mask_1 = jnp.greater(mc, self.mc_wilt)
    flux = flux.at[:].set(jnp.where(_mask_1, (mc - self.mc_wilt) * self.dt, flux[:]))
    self = eqx.tree_at(lambda m: (m.mc,), self, (mc,))
    (temp,) = self.compute_temperature()
    self = eqx.tree_at(lambda m: (m.temp,), self, (temp,))
    return (flux, eqx.tree_at(lambda m: (m.mc, m.temp), self, (mc, temp)))

### Why `eqx.tree_at` appears and why it appears twice

#### Background: Equinox modules are immutable

In JAX, all arrays are **immutable**, you cannot modify them in place.
`eqx.Module` extends this to the entire class: once a `DemoSoil` instance
is created, none of its attributes can be reassigned directly.

```python
# This is illegal inside a JIT-compiled method
self.mc = new_mc          # AttributeError: eqx.Module is frozen
self.mc = self.mc.at[0].set(1.0)  # same problem
```

The solution is `eqx.tree_at`, which **creates a new instance** of the
module with the specified leaves replaced, leaving everything else unchanged:

```python
self = eqx.tree_at(lambda m: (m.mc,), self, (mc,))
# reads as: "give me a copy of self where m.mc is replaced by mc"
```

---

#### Why it appears the first time before `compute_temperature`

```python
mc = mc.at[:].set(jnp.where(_mask_1, (mc - self.mc_wilt) * self.dt, flux[:]))
self = eqx.tree_at(lambda m: (m.mc,), self, (mc,))   # <- first update
(temp,) = self.compute_temperature()
```

`compute_temperature` reads `self.mc` internally:

```python
def compute_temperature(self):
    for ji in range(0, self.kjpindex):
        self.temp[ji] = self.temp[ji] + 0.1 * self.dt * self.mc[ji]
```

If `self.mc` is not updated **before** calling `compute_temperature`, it
would use the **old** soil moisture values, the ones from before
`compute_moisture` ran. The `eqx.tree_at` call propagates the updated `mc`
into `self` so that `compute_temperature` sees the correct, already-updated
values when it reads `self.mc`.

This is the JAX equivalent of what in NumPy would simply be:
```python
self.mc = new_mc          # mutate in place
self.compute_temperature() # naturally sees updated self.mc
```
Because mutation is not allowed, the update must be made explicit via
`eqx.tree_at` before any downstream method that depends on the updated value.

---

#### Why it appears the second time, after `compute_temperature`

```python
(temp,) = self.compute_temperature()
self = eqx.tree_at(lambda m: (m.mc, m.temp), self, (mc, temp))  # <- second update
return (flux, ...)
```

`compute_temperature` cannot mutate `self.temp` directly, either it returns
the updated `temp` as a value. The second `eqx.tree_at` writes both `mc`
**and** `temp` back into `self` at the same time before the return, so the
caller receives a fully updated module.

Updating both in a single call:
```python
eqx.tree_at(lambda m: (m.mc, m.temp), self, (mc, temp))
```
is more efficient than two separate calls because it traverses the pytree
only once.

### What JaxConverter changed

| Original (NumPy) | Converted (JAX) | Rule |
|---|---|---|
| `class DemoSoil:` | `class DemoSoil(eqx.Module):` | `AutoDiff` — class restructuring |
| `np.zeros(10)` | `jnp.zeros(10)` | `AutoDiff` — library alias |
| `self.kjpindex: int` | `kjpindex: int = eqx.field(static=True)` | `AutoDiff` — static field |
| `for ji in range(...):` | `lax.scan(_scan_body_0, ...)` | `JaxConverter` — loop lowering |
| `x if cond else y` | `jnp.where(cond, x, y)` | `JaxConverter` — value select |
| `if cond: ... else: ...` | `lax.cond(cond, ...)` | `JaxConverter` — stateful branch |
| `self.mc[ji] = val` | `mc = mc.at[ji].set(val)` | `JaxConverter` — functional update |

---
## 6. Run the converted JAX module

Verify the JAX version produces numerically identical results to the NumPy version.

In [11]:
import importlib.util
import jax.numpy as jnp
import jax
jax.config.update('jax_enable_x64', True)
# Dynamically import the generated JAX module
spec = importlib.util.spec_from_file_location(
    "global_module_demo_soil_jax",
    jax_global,
)
jax_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(jax_module)

DemoSoilJax = jax_module.DemoSoil_eqx

# Instantiate the JAX model
model_jax = DemoSoilJax()

DemoSoilJax = jax_module.DemoSoil_eqx

model_jax = DemoSoilJax()

flux = np.zeros((model_jax.kjpindex,), dtype=np.float64)
flux_jax = jnp.asarray(flux)
flux_jax, model_jax = model_jax.compute_moisture(flux_jax)

logger.info("JAX outputs:")
logger.info(f"  mc   (soil moisture) : {jnp.round(model_jax.mc, 4)}")
logger.info(f"  flux (drainage)      : {jnp.round(flux_jax, 4)}")
logger.info(f"  temp (temperature)   : {jnp.array(model_jax.temp).round(2)}")

[INFO] JAX outputs:

[INFO]   mc   (soil moisture) : [0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4]

[INFO]   flux (drainage)      : [540. 540. 540. 540. 540. 540. 540. 540. 540. 540.]

[INFO]   temp (temperature)   : [72. 72. 72. 72. 72. 72. 72. 72. 72. 72.]

In [12]:
# Numerical equivalence check
atol = 1e-8
rtol = 1e-5
mc_match   = np.allclose(ref_mc,   np.array(model_jax.mc),   atol=atol, rtol=rtol)
flux_match = np.allclose(ref_flux, np.array(flux_jax), atol=atol, rtol=rtol)
temp_match = np.allclose(ref_temp, np.array(model_jax.temp), atol=atol, rtol=rtol)

logger.info(f"Numerical equivalence (absolute tolerance={atol}, relative tolerance={rtol}):")
logger.info(f"  mc   : {'✓ PASS' if mc_match   else '✗ FAIL'}")
logger.info(f"  flux : {'✓ PASS' if flux_match else '✗ FAIL'}")
logger.info(f"  temp : {'✓ PASS' if temp_match else '✗ FAIL'}")

if all([mc_match, flux_match]):
    logger.info("\n✓ JAX and NumPy versions produce identical results.")
else:
    logger.info("\n✗ Mismatch detected — check the conversion output above.")

[INFO] Numerical equivalence (absolute tolerance=1e-08, relative tolerance=1e-05):

[INFO]   mc   : ✓ PASS

[INFO]   flux : ✓ PASS

[INFO]   temp : ✓ PASS

[INFO] 
✓ JAX and NumPy versions produce identical results.

---
## 7. Explore the three modes

In [13]:
# Run all three modes and compare output file suffixes

for mode in ["jax", "fwd", "bwd"]:
    ad = AutoDiff(config_path=None, mode=mode)
    ad.transform(
        class_file=global_module_path,
        main_file=main_path,
    )

logger.info("\nAll generated files:")
for f in sorted(Path(OUTPUT_DIR).glob("*.py")):
    size = f.stat().st_size
    logger.info(f"  {f.name:<55} ({size:>6} bytes)")

/tmp/ipykernel_2249576/1989317830.py:4: UserWarning: No logger provided; using default Logger()
  ad = AutoDiff(config_path=None, mode=mode)


╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: AutoDiff                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🚀 Node Start ─────────────────────────────────────────────────╮
│ Entering node: Create Module                                                                                    │
│ Function: transform(main_file='/tmp/fgpt_autodiff_demo_whu5k4hc/main_demo_soil.py',                             │
│ class_file='/tmp/fgpt_autodiff_demo_whu5k4hc/global_module_demo_soil.py', routine_dir=None)                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: JaxConverter                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Transforming procedure: compute_temperature

...

[INFO]   global_module_demo_soil_jax.py                          (  1331 bytes)

[INFO]   main_demo_soil.py                                       (   232 bytes)

[INFO]   main_demo_soil_b.py                                     (  1055 bytes)

[INFO]   main_demo_soil_d.py                                     (  1055 bytes)

[INFO]   main_demo_soil_jax.py                                   (  1065 bytes)

In [14]:
# Compare the class declaration across the three modes
# All three should inherit from eqx.Module
# bwd should use checkpointed while loops
# THE SAME filing name as Tapenade
# d - fwd
# b - bwd 
for mode in ["jax", "d", "b"]:
    fpath = Path(OUTPUT_DIR) / f"global_module_demo_soil_{mode}.py"
    if fpath.exists():
        lines = fpath.read_text().splitlines()
        # Show first 15 lines — class declaration and imports
        logger.info(f"global_module_demo_soil_{mode}.py — first 15 lines")
        logger.info("\n".join(lines[:15]))

[INFO] global_module_demo_soil_jax.py — first 15 lines

[INFO] #!/usr/bin/env python3
from jax import jit, lax
import equinox as eqx
import jax.numpy as jnp
import jax
import numpy as np

class DemoSoil_eqx(eqx.Module):
    kjpindex: int = eqx.field(static=True)
    dt: float
    mc_sat: float
    mc_wilt: float
    mc: jnp.ndarray
    temp: jnp.ndarray

[INFO] global_module_demo_soil_d.py — first 15 lines

[INFO] #!/usr/bin/env python3
from jax import jit, lax
import equinox as eqx
import jax.numpy as jnp
import jax
import numpy as np

class DemoSoil_eqx(eqx.Module):
    kjpindex: int = eqx.field(static=True)
    dt: float
    mc_sat: float
    mc_wilt: float
    mc: jnp.ndarray
    temp: jnp.ndarray

[INFO] global_module_demo_soil_b.py — first 15 lines

[INFO] #!/usr/bin/env python3
from jax import jit, lax
import equinox as eqx
import jax.numpy as jnp
import jax
import numpy as np

class DemoSoil_eqx(eqx.Module):
    kjpindex: int = eqx.field(static=True)
    dt: float
    mc_sat: float
    mc_wilt: float
    mc: jnp.ndarray
    temp: jnp.ndarray

### Mode comparison

| Mode | Output suffix | While loop style | Use case |
|---|---|---|---|
| `jax` | `_jax.py` | `eqx.internal.while_loop` | XLA-compiled inference |
| `fwd` | `_d.py` | `eqx.internal.while_loop` | Forward-mode AD (planned) |
| `bwd` | `_b.py` | Checkpointed while loop | Reverse-mode AD (planned) |

> **Note:** All three modes produce structurally valid `eqx.Module` subclasses.
> `jax.grad` / `jax.jvp` / `jax.vjp` call sites are not yet emitted,
> they are planned once the differentiation input specification interface is defined.

---
## 8. Cleanup

In [15]:
# Remove the generated files
shutil.rmtree(OUTPUT_DIR)

# Remove the module from sys.path
if OUTPUT_DIR in sys.path:
    sys.path.remove(OUTPUT_DIR)

logger.info("Cleanup complete.")

[INFO] Cleanup complete.

---
## Summary

| Step | What happened |
|---|---|
| Wrote dummy NumPy class | `DemoSoil` with loop, conditional, in-place update |
| Ran `AutoDiff.transform()` | Produced `_jax.py`, `_d.py`, `_d.py` |
| Inspected AST diff | Saw exact transformations applied by `JaxConverter` |
| Verified numerics | JAX and NumPy outputs match within `1e-5` |
| Compared three modes | `jax` / `fwd` / `bwd` differ in while-loop strategy |

### Next steps

- See `Test_Isolator.ipynb` — isolate a Fortran subroutine before converting
- On Spirit: replace `global_module_path` with a real transpiled file from `fgpt isolate --f2py True`
- To use the full pipeline end-to-end:
```bash
fgpt isolate --target_module hydrol --f2py True ...
fgpt autodiff --config_path template.yaml fgpt autodiff --class_file hydrol/hydrol_soil/global_module_hydrol_soil.py ...
    
```